# 04 — Contribution 3: Post-detection investigation module (ProvSAGE-C3)

This notebook implements and validates **ProvSAGE's investigation pipeline**, a post-detection investigation module that compresses STGAN's raw flag set into a compact attack-neighbourhood subgraph.

## Pipeline
1. **Type filtering** — keep only the entity types observed in the confirmed-malicious anchor population (CDM18 types 2 and 6 on DARPA TRACE).
2. **Multi-window temporal scoping** — find the top-5 dense 600-second windows on the median timestamps of the filtered anchors.
3. **1-hop topological expansion** — expand each window's anchors by one hop in the provenance graph.
4. **Compactness-based ranking** — rank candidate subgraphs by size; the most compact subgraph is the predicted attack neighbourhood.

The algorithm consults only inference-time signals (entity type, edge timestamps, provenance-graph topology). Ground-truth labels are used solely for validation.

## Prerequisites
- `03_C2_architectures.ipynb` completed (provides `flag_mask.npy` from the GraphSAGE+MHA eval).
- `matplotlib`, `networkx`, `scipy` available.

## Expected outcome on DARPA TRACE
- Candidate subgraph of ≈ 343 nodes (compression ≈ 3,500× of the full graph).
- Recovers the majority of STGAN's correctly-flagged anchors plus additional confirmed-malicious entities via 1-hop expansion.

## Outputs
- `eval/c3_outputs_temporal/v4_candidate_subgraph.npy`
- `eval/c3_outputs_temporal/v4_metrics.json`
- `eval/c3_outputs_temporal/v4_attack_neighborhood.{png,pdf}`

In [1]:
print("=== Notebook 04 — Contribution 3: ProvSAGE Investigation Pipeline ===")

=== Notebook 04 — Contribution 3: ProvSAGE Investigation Pipeline ===


In [2]:
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("STGAN_ROOT", Path.cwd().resolve().parent))
assert (PROJECT_ROOT / "code" / "eval").exists(), f"PROJECT_ROOT incorrect : {PROJECT_ROOT}"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = /home/jovyan/stgan/work/STGAN-master


In [ ]:
import subprocess

# c3_visualize_attack_neighborhood.py implémente le pipeline décrit ci-dessus
# (filtrage par type -> scoping temporel -> expansion 1-hop -> ranking par compacité)
# et sauvegarde la figure + les métriques dans results/c3_outputs_visualization/.
script = f"""
cd {PROJECT_ROOT}/code/eval
PYTHONUNBUFFERED=1 PYTHONPATH={PROJECT_ROOT}/code \\
python -u c3_visualize_attack_neighborhood.py
"""
subprocess.run(["bash", "-c", script], check=True)
print("\n✓ ProvSAGE-C3 terminé")

In [ ]:
from IPython.display import Image, display
import json

out_dir = PROJECT_ROOT / "code" / "eval" / "c3_outputs_visualization"
png = out_dir / "attack_neighborhood.png"
metrics_path = out_dir / "metrics.json"

if png.exists():
    display(Image(filename=str(png)))
else:
    print(f"⚠ PNG non trouvé à {png} — une copie pré-calculée est disponible dans results/c3_outputs_visualization/")

if metrics_path.exists():
    print(json.dumps(json.load(open(metrics_path)), indent=2))